In [0]:
# Silver layer: order_items (B2B order line facts)
# Source: b2b_ecommerce.bronze.order_items (append-only, connector-managed,
#         already typed: bigint order_item_id / order_id, int product_id /
#         supplier_id / quantity / line_number, decimal(10,2) unit_price,
#         decimal(4,1) discount_pct / tax_rate, decimal(12,2) line_total,
#         string warehouse, timestamps)
# Grain:  one row per order_item_id AND version (SCD Type 2); the current
#         row is the one with __END_AT IS NULL. Every bronze row is already
#         unique by order_item_id AND by (order_id, line_number)
#         (30,240 rows = 30,240 ids), so no dedup is needed - the AUTO CDC
#         merge below is what keeps it unique if a changed line is ever
#         re-synced from the source.
# Profile (2026-11-12, 30,240 rows): warehouse arrives in only 3 clean
#         values ('BsAs-Central', 'Rosario-Sur', 'Cordoba-Norte') - trim only,
#         no case dirt. line_number 1-8, unique per order, never null.
#         discount_pct 0.0-20.0, tax_rate 10.5-21.0 - plausible, kept as-is.
#         Measure dirt: 491 rows have quantity <= 0 (min -50, incl. 0 where
#         the true qty is recoverable from line_total / unit_price) and 726
#         rows have unit_price NULL (415) / negative (190, min -438.52) /
#         zero (121). The dirt is disjoint: no row has both bad. line_total
#         is ALWAYS trustworthy: on the 29,023 clean rows it equals exactly
#         quantity * unit_price (simple extension, discount/tax NOT included),
#         and on dirty rows it holds the TRUE extension (e.g. quantity 0 but
#         line_total = 9 * price) - so the corrupted columns are quantity /
#         unit_price, NOT line_total. Both untrustworthy measures are
#         NULLified and the row KEPT (line_total preserves the business
#         value); excluding a line from a revenue report is a GOLD decision
#         (same policy as silver.orders shipping_cost / dates).
# FKs verified clean: 0 orphans vs current silver.orders, silver.products
#         and silver.suppliers, and every (product_id, supplier_id) pair
#         also exists in silver.supplier_products (0 orphans).
# SCD:    30,210 of 30,240 rows have updated_at > created_at, i.e. order
#         lines ARE updated at the source. A changed line re-synced into
#         bronze appends a new version under Type 2 and the history is kept
#         (same choice as silver.orders / customers / products / suppliers).
# Idempotency: AUTO CDC keyed by order_item_id MERGEs into b2b_ecommerce
#         .silver.order_items - identical re-sends collapse (updated_at is
#         excluded from history tracking), re-ingestion never duplicates.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_ORDER_ITEMS = "b2b_ecommerce.bronze.order_items"
# current dims/facts live in the silver layer (same pipeline datasets)
SILVER_ORDERS = "orders"
SILVER_PRODUCTS = "products"
SILVER_SUPPLIERS = "suppliers"


def clean_text(name, lower=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned stream of bronze.order_items (+ FK match flags vs current
#    silver orders / products / suppliers)
# ---------------------------------------------------------------------------
@dp.temporary_view(name="order_items_clean_stream")
# Data quality validation (only checks that can actually catch bad rows).
# Drop-level: without the merge key or the CDC sequence the row cannot
# be merged at all - quarantine it out of silver; the raw value stays in
# bronze for inspection. Everything else is kept and MARKED, never
# dropped - silver preserves business entities and flags problems, gold
# owns the reporting rules.
@dp.expect_or_drop("order_item_id_not_null", "order_item_id IS NOT NULL")
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")
# Referential integrity: every order line must reference a CURRENT silver
# order, product and supplier - a missing link fails the whole update
# instead of silently landing in silver (hard stop is intended, same
# policy as silver.orders / suppliers; verified clean today: 0 orphans on
# all three sides). No separate not-null warn checks: a NULL id never
# matches the join, so the exists checks already catch it.
@dp.expect_or_fail("order_id_exists", "_order_id_match IS NOT NULL")
@dp.expect_or_fail("product_id_exists", "_product_id_match IS NOT NULL")
@dp.expect_or_fail("supplier_id_exists", "_supplier_id_match IS NOT NULL")
# Warn-level: the row is kept and the problem only recorded in the event
# log (same policy as silver.orders). These fire only on NEW dirt from the
# source - all current rows pass after cleaning.
@dp.expect("warehouse_valid", "warehouse IN ('BsAs-Central', 'Rosario-Sur', 'Cordoba-Norte')")
@dp.expect("discount_pct_valid", "discount_pct >= 0 AND discount_pct <= 100")
@dp.expect("tax_rate_valid", "tax_rate >= 0")
# Kept-but-invalid measures (491 non-positive quantity + 726 null/negative/
# zero unit_price) - counted here so they stay visible in the pipeline
# quality metrics; the row itself is kept, the value is NULLified below.
@dp.expect("quantity_valid", "quantity > 0")
@dp.expect("unit_price_valid", "unit_price > 0")
def order_items_clean_stream():
    bronze = spark.readStream.table(BRONZE_ORDER_ITEMS)

    # stream-static LEFT JOINs against CURRENT silver rows: a dim/fact row
    # that exists in bronze but was dropped from silver does not match -
    # the row becomes an orphan and the fail-level expectation stops the
    # update. An expectation is a per-row column expression, so each FK
    # check needs its match flag as a column.
    orders = (
        spark.read.table(SILVER_ORDERS)
        .filter("__END_AT IS NULL")
        .select(F.col("order_id").alias("_order_id_match"))
        .distinct()
    )
    products = (
        spark.read.table(SILVER_PRODUCTS)
        .filter("__END_AT IS NULL")
        .select(F.col("product_id").alias("_product_id_match"))
        .distinct()
    )
    suppliers = (
        spark.read.table(SILVER_SUPPLIERS)
        .filter("__END_AT IS NULL")
        .select(F.col("supplier_id").alias("_supplier_id_match"))
        .distinct()
    )

    bronze = bronze.join(
        orders,
        bronze["order_id"] == orders["_order_id_match"],
        "left",
    ).join(
        products,
        bronze["product_id"] == products["_product_id_match"],
        "left",
    ).join(
        suppliers,
        bronze["supplier_id"] == suppliers["_supplier_id_match"],
        "left",
    )

    return bronze.select(
        F.col("order_item_id").cast("long").alias("order_item_id"),
        F.col("order_id").cast("long").alias("order_id"),
        F.col("product_id").cast("int").alias("product_id"),
        F.col("supplier_id").cast("int").alias("supplier_id"),
        # untrustworthy measures are NULLified, the line itself is kept:
        # 491 rows have quantity <= 0 (min -50) - line_total still holds
        # the TRUE extension, so the business value is not lost
        F.when(F.col("quantity") > 0, F.col("quantity"))
        .otherwise(F.lit(None))
        .cast("int")
        .alias("quantity"),
        # 726 rows have a NULL (415) / negative (190, min -438.52) / zero
        # (121) unit_price - zero is provably wrong because line_total > 0
        # on every such row
        F.when(F.col("unit_price") > 0, F.col("unit_price"))
        .otherwise(F.lit(None))
        .cast("decimal(10,2)")
        .alias("unit_price"),
        # line_number 1-8, unique per order, never null - kept as-is
        F.col("line_number").cast("int").alias("line_number"),
        # discount_pct 0.0-20.0 and tax_rate 10.5-21.0 - plausible ranges,
        # kept as-is
        F.col("discount_pct").cast("decimal(4,1)").alias("discount_pct"),
        F.col("tax_rate").cast("decimal(4,1)").alias("tax_rate"),
        # line_total is the one fully consistent measure: it equals
        # quantity * unit_price on all 29,023 clean rows and holds the
        # TRUE extension on dirty rows - kept AS-IS (simple extension,
        # discount/tax NOT included, gold applies them if needed)
        F.col("line_total").cast("decimal(12,2)").alias("line_total"),
        # only 3 clean values arrive; trim guards against new dirt
        clean_text("warehouse"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        F.col("updated_at").cast("timestamp").alias("updated_at"),
        # internal FK flags, never land in silver
        F.col("_order_id_match"),
        F.col("_product_id_match"),
        F.col("_supplier_id_match"),
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.order_items
#    Rows are MERGEd by order_item_id (SCD Type 2): identical re-sends
#    update in place, a genuinely changed row opens a new version
#    (__START_AT / __END_AT validity columns), the history is kept.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="order_items",
    comment=(
        "Silver order items: cleaned and standardized; one row per "
        "order_item_id and version (SCD Type 2 merge - latest updated_at "
        "wins, history kept)"
    ),
)

dp.create_auto_cdc_flow(
    target="order_items",
    source="order_items_clean_stream",
    keys=["order_item_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2,
    # updated_at itself is excluded from history tracking: an identical
    # re-send (same values, newer updated_at) updates in place instead of
    # opening a duplicate version (same fix as silver.products / orders).
    track_history_except_column_list=["updated_at"],
    # internal FK flags must not land in the target table
    except_column_list=[
        "_order_id_match",
        "_product_id_match",
        "_supplier_id_match",
    ],
)